# Model 3 — EfficientNet-B0 (ImageNet transfer learning)

**Comparative Deep Learning Framework for Localized Road Damage Classification (RDD2022)**

Owner: Student 3 · Model: `efficientnet_b0`

A compound-scaled architecture in which depth, width and resolution are scaled jointly (Tan & Le, 2019). It represents the accuracy-per-parameter-optimised end of the comparison (synopsis §4.3).

---

## How this notebook relates to the other two

Three notebooks exist, one per architecture: `custom_cnn`, `resnet50`,
`efficientnet_b0`. **Sections 1–7 are byte-for-byte identical across all three.**
That is deliberate and it is the methodological core of the project: the
comparison is only valid if preprocessing, filtering, splitting and augmentation
are held constant and the architecture is the only thing that varies (§15.1).

Because the split is generated from a **fixed seed (42)** and the extraction
filters are fixed constants, running Sections 1–7 on three different machines
produces identical data. Section 7 ends with a **fingerprint check** that proves
it — each person records the fingerprint, and all three must match before any
result is comparable.

Sections 8 onward are specific to this model alone.

### Run order
1. Sections 1–7 — shared preprocessing (~45 min first time, seconds afterwards)
2. **Verify the fingerprint matches your teammates'** before training
3. Sections 8–11 — train, evaluate, interpret this one model
4. Section 12 — merge results once all three are done

## 1 · Configuration

Every value under *fixed protocol* is held constant across all three
architectures. **Do not change any of these** — a single differing value
invalidates the comparison. Only `MODEL_NAME` and the block in Section 8 differ
between notebooks.

In [ ]:
from pathlib import Path

MODEL_NAME = "efficientnet_b0"          # the ONLY identifier that differs between the three notebooks

PROJECT = Path.cwd()           # notebook must sit beside the Dataset folder
CFG = {
    "dataset_root": PROJECT / "Dataset",
    "data_dir":     PROJECT / "data",
    "reports_dir":  PROJECT / "reports",
    "runs_dir":     PROJECT / "runs",

    # ---- fixed experimental protocol (§15.1) — IDENTICAL IN ALL THREE -----
    "seed": 42,
    "classes": ["D00", "D10", "D20", "D40"],
    "image_size": 224,
    "batch_size": 32,                 # bounded by 6 GB VRAM; drop to 16 on OOM
    "epochs": 40,
    "early_stopping_patience": 8,
    "weight_decay": 1e-4,
    "label_smoothing": 0.05,
    "amp": True,

    # ---- patch extraction (§14.3–14.4) -----------------------------------
    "margin": 0.15,                   # proportional context around each box
    "min_side": 16,                   # chosen by per-class retention analysis
    "max_aspect": 20,                 # loose: D10 is inherently elongated

    # ---- splitting (§16) --------------------------------------------------
    "group_size": 5,                  # consecutive frames per split unit
    "ratios": (0.70, 0.15, 0.15),

    # ---- normalisation for the from-scratch model only --------------------
    "dataset_mean": [0.4854, 0.4797, 0.4611],
    "dataset_std":  [0.2008, 0.1868, 0.1806],
}

# Windows + Jupyter: DataLoader worker processes deadlock inside a notebook
# kernel, because worker startup re-imports the parent module and a notebook has
# no module file to re-import. 0 keeps loading in the main process: slower, but
# it does not hang. Raise only if you convert this to a .py script.
NUM_WORKERS = 0

for k in ("data_dir", "reports_dir", "runs_dir"):
    CFG[k].mkdir(parents=True, exist_ok=True)

INDEX_CSV    = CFG["data_dir"] / "index.csv"
REJECTS_CSV  = CFG["data_dir"] / "index_rejects.csv"
SPLITS_CSV   = CFG["data_dir"] / "splits_main.csv"
PATCH_ROOT   = CFG["data_dir"] / "patches_main"
MANIFEST_CSV = CFG["data_dir"] / "manifest_main.csv"

print("model   :", MODEL_NAME)
print("project :", PROJECT)
print("dataset :", CFG["dataset_root"],
      "->", "FOUND" if CFG["dataset_root"].is_dir() else "*** MISSING ***")

## 2 · Imports and environment check

One import cell for the whole notebook, so a kernel restart needs only this cell
re-run rather than hunting for whichever cell defined `plt` or `Image`.

If `cuda available` prints `False`, stop. Check `torch CUDA` — if it is `None`
you installed the CPU-only wheel and need to reinstall from pytorch.org using the
CUDA `--index-url`.

In [ ]:
import json, time, random, platform, hashlib, warnings
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T, models
from sklearn.metrics import (f1_score, accuracy_score, confusion_matrix,
                             classification_report, precision_recall_fscore_support)

%matplotlib inline
warnings.filterwarnings("ignore", category=UserWarning)

print("torch         :", torch.__version__)
print("torch CUDA    :", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    _p = torch.cuda.get_device_properties(0)
    print("gpu           :", _p.name)
    print("vram          :", round(_p.total_memory / 1e9, 2), "GB")
    print("compute cap   :", f"{_p.major}.{_p.minor}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = True      # autotune conv algorithms
set_seed(CFG["seed"])
print("seed set      :", CFG["seed"])

## 3 · Stage 1 — Annotation parsing  ·  §14.2

Walks the country folders, parses every Pascal VOC XML, applies integrity checks,
and produces `index.csv`: one row per damage instance. This is the single source
of truth for every downstream stage.

Only the `train` folders are read. The released `test` folders contain **no
annotations** — labels were withheld for the CRDDC'2022 leaderboard — so the
train/val/test split is carved entirely out of `train`.

Subset discovery searches at any depth, because the archives unpack
inconsistently: some produce `Dataset/Norway/train/`, others
`Dataset/Norway/Norway/train/`.

In [ ]:
def find_subsets(root):
    """Discover country subsets at any depth, with either annotation layout."""
    found = {}
    for img_dir in sorted(root.rglob("images")):
        if not img_dir.is_dir() or img_dir.parent.name != "train":
            continue                              # train only; test has no labels
        base = img_dir.parent
        ann = base / "annotations" / "xmls"
        if not ann.is_dir():
            ann = base / "annotations"
        if not ann.is_dir() or not any(ann.glob("*.xml")):
            continue
        name = base.parent.name                   # collapse Norway/Norway -> Norway
        if name in found:
            name = f"{base.parent.parent.name}_{name}"
        found[name] = (img_dir, ann)
    return found


def parse_one_xml(xml_path, img_dir, country):
    """Parse one VOC file. Returns (instance_rows, reject_rows)."""
    rows, rej = [], []
    try:
        root = ET.parse(xml_path).getroot()
    except ET.ParseError as e:
        return [], [(xml_path.name, "malformed_xml", str(e)[:60])]

    el = root.find("filename")
    fname = el.text.strip() if el is not None and el.text else xml_path.stem + ".jpg"
    img_path = img_dir / fname
    if not img_path.exists():
        alt = img_dir / (xml_path.stem + ".jpg")
        if not alt.exists():
            return [], [(xml_path.name, "missing_image", fname)]
        img_path = alt

    size = root.find("size")
    try:
        img_w = int(float(size.find("width").text))
        img_h = int(float(size.find("height").text))
    except (AttributeError, TypeError, ValueError):
        return [], [(xml_path.name, "missing_size", "")]
    if img_w <= 0 or img_h <= 0:
        return [], [(xml_path.name, "bad_size", f"{img_w}x{img_h}")]

    objects = root.findall("object")
    if not objects:
        # ~31% of RDD2022 train images carry no annotation. Expected, not an error.
        return [], [(xml_path.name, "no_objects", "")]

    for i, obj in enumerate(objects):
        n = obj.find("name")
        if n is None or not n.text:
            rej.append((xml_path.name, "missing_class", f"obj{i}")); continue
        cls = n.text.strip()
        bb = obj.find("bndbox")
        if bb is None:
            rej.append((xml_path.name, "missing_bndbox", f"obj{i}")); continue
        try:
            x0, y0 = int(float(bb.find("xmin").text)), int(float(bb.find("ymin").text))
            x1, y1 = int(float(bb.find("xmax").text)), int(float(bb.find("ymax").text))
        except (AttributeError, TypeError, ValueError):
            rej.append((xml_path.name, "bad_coords", f"obj{i}")); continue
        if x1 <= x0 or y1 <= y0:
            rej.append((xml_path.name, "nonpositive_area", f"obj{i}")); continue

        # Clamp rather than discard: a few boxes overhang the frame edge by a
        # pixel or two, which is a rounding artefact and not bad data.
        cx0, cy0 = max(0, x0), max(0, y0)
        cx1, cy1 = min(img_w, x1), min(img_h, y1)
        if cx1 <= cx0 or cy1 <= cy0:
            rej.append((xml_path.name, "out_of_bounds", f"obj{i}")); continue

        bw, bh = cx1 - cx0, cy1 - cy0
        rows.append(dict(
            instance_id=f"{country}__{img_path.stem}__{i}", country=country,
            source_image=str(img_path), image_stem=img_path.stem,
            img_w=img_w, img_h=img_h, class_code=cls,
            xmin=cx0, ymin=cy0, xmax=cx1, ymax=cy1,
            box_w=bw, box_h=bh, box_area=bw * bh,
            aspect_ratio=round(bw / bh, 4),
            clamped=int((cx0, cy0, cx1, cy1) != (x0, y0, x1, y1)),
        ))
    return rows, rej


SUBSETS = find_subsets(CFG["dataset_root"])
COUNTRIES = sorted(SUBSETS)
print(f"found {len(COUNTRIES)} subsets:")
for c in COUNTRIES:
    print(f"  {c:20s} {sum(1 for _ in SUBSETS[c][1].glob('*.xml')):6d} xmls")

### 3.1 · Run the parser

Cached: if `index.csv` already exists it is loaded rather than rebuilt. Delete
the file to force a re-parse.

In [ ]:
if INDEX_CSV.exists():
    index_df = pd.read_csv(INDEX_CSV)
    print(f"loaded cached index: {len(index_df):,} instances from "
          f"{index_df.source_image.nunique():,} images")
else:
    all_rows, all_rej = [], []
    for country in COUNTRIES:
        img_dir, ann_dir = SUBSETS[country]
        xmls = sorted(ann_dir.glob("*.xml"))
        print(f"{country:18s} {len(xmls):6d} annotation files")
        for xp in tqdm(xmls, desc=country, leave=False):
            r, j = parse_one_xml(xp, img_dir, country)
            all_rows.extend(r); all_rej.extend(j)

    if not all_rows:
        raise SystemExit("No instances parsed. Check CFG['dataset_root'] and folder layout.")

    index_df = pd.DataFrame(all_rows)
    index_df.to_csv(INDEX_CSV, index=False)
    pd.DataFrame(all_rej, columns=["xml", "reason", "detail"]).to_csv(REJECTS_CSV, index=False)

    print(f"\n{len(index_df):,} instances from {index_df.source_image.nunique():,} images")
    print("\nReject reasons (report these in Phase 2):")
    for reason, n in Counter(x[1] for x in all_rej).most_common():
        print(f"  {reason:18s} {n:,}")

## 4 · Stage 1b — Class distribution and the class-set decision  ·  §13.4

RDD2022's files contain more than the four documented damage types. The
non-canonical codes are road markings, fixtures and repairs rather than damage:

| Code | Meaning | Decision |
|---|---|---|
| D43 | Crosswalk blur | Drop — faded marking, not damage |
| D44 | White line blur | Drop — faded marking, not damage |
| D50 | Manhole / utility cover | Drop — street furniture |
| Repair | Maintenance patch | Drop — the *absence* of damage |
| D01, D11 | Construction joint parts | Drop — engineered joints |
| Block crack, D0w0 | 3 and 1 instances | Drop — negligible; `D0w0` is a typo for `D00` |

Keeping only D00/D10/D20/D40 yields **55,006 instances**, which reconciles exactly
with the "more than 55,000" figure published in the RDD2022 data article. That
arithmetic is the justification for the class set — not a convention.

Note for the report: the non-canonical codes sit almost entirely in Japan and
India, so dropping them is **not** a geographically neutral operation.

In [ ]:
xt = pd.crosstab(index_df.class_code, index_df.country,
                 margins=True, margins_name="TOTAL").sort_values("TOTAL", ascending=False)
xt.to_csv(CFG["reports_dir"] / "class_by_country.csv")
print("INSTANCES: class x country")
print(xt.to_string())

kept = index_df[index_df.class_code.isin(CFG["classes"])]
print(f"\ncanonical four: {len(kept):,} instances "
      f"({len(kept)/len(index_df)*100:.1f}% of all parsed)")
print("   reconciles with the published 'more than 55,000' figure")

per_img = index_df.groupby("source_image").size()
print(f"\nLEAKAGE EXPOSURE")
print(f"  instances per image: mean {per_img.mean():.2f}, max {per_img.max()}")
print(f"  images with >1 instance: {(per_img>1).sum():,} ({(per_img>1).mean()*100:.1f}%)")
print("  ^ each is a leakage vector under naive patch-level splitting (§16)")

fig, ax = plt.subplots(1, 2, figsize=(15, 4.2))
vc = index_df.class_code.value_counts()
vc.plot.bar(ax=ax[0], color="#4a6fa5"); ax[0].set_title("All codes present in the files")
for i, v in enumerate(vc.values):
    ax[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=7)
sub = xt.drop(index="TOTAL", errors="ignore").drop(columns="TOTAL", errors="ignore")
sub.loc[CFG["classes"]].T.plot.bar(ax=ax[1], stacked=True, colormap="tab20")
ax[1].set_title("Kept classes, by country")
plt.tight_layout(); plt.savefig(CFG["reports_dir"] / "distribution.png", dpi=150); plt.show()

## 5 · Stage 2 — Leakage-resistant splitting  ·  §16

**This is the methodological core, and the reason all three notebooks must run
identical preprocessing.**

Splitting happens at the **source-image level, before any patch exists**, so a
patch can never be assigned a partition independently of its parent image.

`group_size = 5` goes further: consecutive frames within a country are bucketed
together. Road imagery is captured as a sequence from a moving vehicle, so
adjacent filenames often show the same physical crack a metre further along.
Image-level splitting alone does not catch that.

61.7% of annotated images contain more than one instance, so under naive
patch-level splitting nearly two-thirds of the data would have siblings scattered
across train and test. The resulting scores would be inflated by scene
recognition rather than damage morphology.

Because the seed is fixed at 42, this produces an identical split on every
machine. **The audit below must print PASS.**

In [ ]:
def build_groups(images, group_size):
    """group_size=1 -> per-image. >1 -> bucket consecutive frames per country."""
    if group_size <= 1:
        return images.image_stem
    mapping = {}
    for country, sub_ in images.groupby("country"):
        for i, stem in enumerate(sorted(sub_.image_stem)):
            mapping[stem] = f"{country}__grp{i // group_size:06d}"
    return images.image_stem.map(mapping)


def stratified_group_split(groups_df, ratios, seed):
    """Assign whole groups to partitions, stratified by (country, dominant class)."""
    rng = np.random.default_rng(seed)
    assign = {}
    for _, sub_ in groups_df.groupby(["country", "dominant_class"]):
        g = sub_.group.tolist(); rng.shuffle(g); n = len(g)
        n_tr, n_va = int(round(n * ratios[0])), int(round(n * ratios[1]))
        if n >= 3:                       # guarantee each partition receives one
            n_tr = max(1, min(n_tr, n - 2))
            n_va = max(1, min(n_va, n - n_tr - 1))
        for i, grp in enumerate(g):
            assign[grp] = "train" if i < n_tr else ("val" if i < n_tr + n_va else "test")
    return assign


def make_splits(index_df, classes, ratios, group_size, seed,
                mode="standard", holdout=None):
    d = index_df[index_df.class_code.isin(classes)]
    dom = (d.groupby(["country", "image_stem"]).class_code
             .agg(lambda s: s.value_counts().idxmax())
             .reset_index().rename(columns={"class_code": "dominant_class"}))
    dom["group"] = build_groups(dom, group_size)

    if mode == "holdout":
        # Held-out country contributes NOTHING to training, validation or
        # model selection. It is the test set, full stop.
        tgt = dom[dom.country == holdout].copy()
        src = dom[dom.country != holdout].copy()
        gdf = (src.groupby("group").agg(country=("country", "first"),
               dominant_class=("dominant_class", "first")).reset_index())
        tot = ratios[0] + ratios[1]
        a = stratified_group_split(gdf, (ratios[0]/tot, ratios[1]/tot, 0.0), seed)
        src["split"] = src.group.map(a).fillna("train")
        src.loc[src.split == "test", "split"] = "train"
        tgt["split"] = "test"
        dom = pd.concat([src, tgt], ignore_index=True)
    else:
        gdf = (dom.groupby("group").agg(country=("country", "first"),
               dominant_class=("dominant_class", "first")).reset_index())
        dom["split"] = dom.group.map(stratified_group_split(gdf, ratios, seed))
    return dom


splits_df = make_splits(index_df, CFG["classes"], CFG["ratios"],
                        CFG["group_size"], CFG["seed"])
splits_df[["country", "image_stem", "group", "dominant_class", "split"]].to_csv(
    SPLITS_CSV, index=False)

n_bad = int((splits_df.groupby("group").split.nunique() > 1).sum())
print("=" * 78)
print(f"LEAKAGE AUDIT: {n_bad} groups spanning multiple partitions -> "
      f"{'PASS' if n_bad == 0 else '*** FAIL — DO NOT CONTINUE ***'}")
print("=" * 78)

print("\nSource images per partition:")
print(splits_df.split.value_counts().to_string())

m = index_df[index_df.class_code.isin(CFG["classes"])].merge(
    splits_df[["country", "image_stem", "split"]], on=["country", "image_stem"])
print("\nInstance-level class balance per partition (%):")
print((pd.crosstab(m.split, m.class_code, normalize="index") * 100).round(1).to_string())
print(f"\n-> {SPLITS_CSV}  (seed={CFG['seed']}, group_size={CFG['group_size']})")

## 6 · Stage 3 — Patch extraction and quality filtering  ·  §14.3–14.4

RDD2022 is a **detection** dataset: find damage in a road photo and box it. This
project is **classification**: given a damage region, name its type. This cell
performs that conversion — each annotated box is cropped from its source photo
and saved as its own image.

Filter thresholds were chosen by measuring **per-class retention**, not by
convention:

| Threshold | D00 | D10 | D20 | D40 | Verdict |
|---|---|---|---|---|---|
| `min_side ≥ 24`, `aspect ≤ 8` | 93% | **69%** | 100% | 82% | Rejected — class-selective |
| `min_side ≥ 16`, `aspect ≤ 20` | 98% | 91% | 100% | 96% | **Adopted** — near-uniform |

Transverse cracks (D10) are inherently elongated — a thin line spanning the road,
often 16 px tall and 400 px wide. A strict aspect limit removes them and almost
nothing else, turning a quality filter into a silent deletion of one class.

Runs once (~40 min) and is cached afterwards. The partition label is **inherited**
from the parent image; a patch never gets its own split decision.

In [ ]:
def extract_patches(index_df, splits_df, cfg, patch_root, manifest_csv):
    d = index_df[index_df.class_code.isin(cfg["classes"])].merge(
        splits_df[["country", "image_stem", "split"]], on=["country", "image_stem"])
    for sp in ("train", "val", "test"):
        for c in cfg["classes"]:
            (patch_root / sp / c).mkdir(parents=True, exist_ok=True)

    rejected = Counter(); records = []
    size, margin = cfg["image_size"], cfg["margin"]

    # Group by source image so each JPEG is decoded ONCE, not once per box.
    # With 2.46 boxes per image this is the difference between 40 min and 2 h.
    for src_img, grp in tqdm(d.groupby("source_image"), desc="extracting"):
        img = cv2.imread(src_img)
        if img is None:
            rejected["unreadable"] += len(grp); continue
        H, W = img.shape[:2]
        for r in grp.itertuples():
            bw, bh = r.box_w, r.box_h
            if min(bw, bh) < cfg["min_side"]:
                rejected["min_side"] += 1; continue
            if max(bw / bh, bh / bw) > cfg["max_aspect"]:
                rejected["aspect"] += 1; continue
            mx, my = int(bw * margin), int(bh * margin)
            crop = img[max(0, r.ymin - my):min(H, r.ymax + my),
                       max(0, r.xmin - mx):min(W, r.xmax + mx)]
            if crop.size == 0:
                rejected["empty_crop"] += 1; continue
            interp = cv2.INTER_AREA if min(crop.shape[:2]) > size else cv2.INTER_CUBIC
            crop = cv2.resize(crop, (size, size), interpolation=interp)
            rel = f"{r.split}/{r.class_code}/{r.instance_id}.jpg"
            cv2.imwrite(str(patch_root / rel), crop, [cv2.IMWRITE_JPEG_QUALITY, 95])
            records.append(dict(patch_path=rel, instance_id=r.instance_id,
                                class_code=r.class_code, country=r.country,
                                split=r.split, image_stem=r.image_stem,
                                orig_w=bw, orig_h=bh,
                                upscaled=int(min(bw, bh) < size)))
    man = pd.DataFrame(records)
    man.to_csv(manifest_csv, index=False)
    return man, rejected


if MANIFEST_CSV.exists():
    manifest_df = pd.read_csv(MANIFEST_CSV)
    print(f"loaded cached manifest: {len(manifest_df):,} patches")
    print(manifest_df.split.value_counts().to_string())
else:
    manifest_df, rejected = extract_patches(index_df, splits_df, CFG,
                                            PATCH_ROOT, MANIFEST_CSV)
    print(f"\n{len(manifest_df):,} patches -> {PATCH_ROOT}")
    print("\nRejected by filter (report these):")
    for k, v in rejected.most_common():
        print(f"  {k:14s} {v:,}")

print("\nPatches: split x class")
print(pd.crosstab(manifest_df.split, manifest_df.class_code, margins=True).to_string())
print(f"\nUpscaled (source < {CFG['image_size']}px): {manifest_df.upscaled.mean()*100:.1f}%")
print("  ^ a genuine limitation: most patches are interpolated, not captured, at 224px.")
print("    It also creates a mild domain shift for the ImageNet-pretrained models,")
print("    whose weights were learned on sharp native-resolution photographs.")

n_bad = int((manifest_df.groupby("image_stem").split.nunique() > 1).sum())
print("\n" + "=" * 78)
print(f"LEAKAGE ASSERTION: {n_bad} source images in >1 partition -> "
      f"{'PASS' if n_bad == 0 else '*** FAIL — DO NOT CONTINUE ***'}")
print("=" * 78)

## 7 · Preprocessing fingerprint  ·  **cross-check with your teammates**

Everything above should have produced identical data on all three machines. This
cell proves it rather than assuming it.

The fingerprint hashes the sorted patch manifest — every patch id, its class and
its partition. If two people get the same fingerprint, they are training on
exactly the same data and their results are directly comparable. If the
fingerprints differ, something upstream diverged (a different dataset copy, an
edited threshold, a changed seed) and the comparison is **invalid** until it is
resolved.

**Record this value and compare it before anyone starts training.** Put it in the
Phase 2 report as a reproducibility artefact.

In [ ]:
def preprocessing_fingerprint(manifest_df, cfg):
    """Stable hash of the exact training data, independent of row order."""
    key = (manifest_df[["instance_id", "class_code", "split"]]
           .sort_values("instance_id").to_csv(index=False))
    protocol = json.dumps({k: str(cfg[k]) for k in
                           ["seed", "classes", "image_size", "min_side",
                            "max_aspect", "margin", "group_size", "ratios"]},
                          sort_keys=True)
    return (hashlib.md5(key.encode()).hexdigest()[:16],
            hashlib.md5(protocol.encode()).hexdigest()[:8])

data_fp, proto_fp = preprocessing_fingerprint(manifest_df, CFG)

print("=" * 78)
print("PREPROCESSING FINGERPRINT — all three notebooks must match")
print("=" * 78)
print(f"  data fingerprint     : {data_fp}")
print(f"  protocol fingerprint : {proto_fp}")
print(f"  patches              : {len(manifest_df):,}")
print(f"  train / val / test   : " +
      " / ".join(f"{(manifest_df.split==s).sum():,}" for s in ("train","val","test")))
print("=" * 78)

json.dump({"model": MODEL_NAME, "data_fingerprint": data_fp,
           "protocol_fingerprint": proto_fp, "n_patches": int(len(manifest_df)),
           "generated": time.strftime("%Y-%m-%d %H:%M")},
          open(CFG["reports_dir"] / f"fingerprint_{MODEL_NAME}.json", "w"), indent=2)
print(f"saved -> reports/fingerprint_{MODEL_NAME}.json")
print("\nShare this with the other two students. Do not train until all three match.")

## 8 · Dataset, augmentation and the efficientnet_b0 architecture

**From here on this notebook differs from the other two.**

### Augmentation — fixed across all three models (§14.7)

Horizontal flip, small affine jitter and mild colour jitter. Vertical flip and
large-angle rotation are deliberately **excluded**: crack orientation is
class-defining, so a longitudinal crack rotated 90° becomes visually identical to
a transverse crack. That augmentation would corrupt the label itself.

Augmentation applies to the training partition only.

### Normalisation

This model uses **ImageNet** normalisation statistics, matching the pretrained weights — identical treatment to ResNet50, so the two transfer-learning models differ only in backbone (§14.8).

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

class PatchDataset(Dataset):
    def __init__(self, manifest_df, root, split, classes, transform):
        d = manifest_df[(manifest_df.split == split) &
                        (manifest_df.class_code.isin(classes))]
        self.paths = d.patch_path.tolist()
        self.countries = d.country.tolist()
        self.cls_to_idx = {c: i for i, c in enumerate(classes)}
        self.labels = [self.cls_to_idx[c] for c in d.class_code]
        self.root, self.transform = Path(root), transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.root / self.paths[i]).convert("RGB")
        return self.transform(img), self.labels[i]


def build_transforms(size, mean, std, train):
    """Identical policy in all three notebooks. Train-only augmentation."""
    if train:
        return T.Compose([
            T.Resize((size, size)),
            T.RandomHorizontalFlip(0.5),          # safe: preserves orientation class
            T.RandomAffine(degrees=7, translate=(.05, .05), scale=(.9, 1.1)),
            T.ColorJitter(brightness=.2, contrast=.2, saturation=.1),
            T.ToTensor(), T.Normalize(mean, std)])
    return T.Compose([T.Resize((size, size)), T.ToTensor(), T.Normalize(mean, std)])


# ---- training regime for THIS model (§14.8) --------------------------------
SPEC = dict(pretrained=True, freeze_epochs=5,
            lr_head=0.001, lr_finetune=0.0001)

# Normalisation follows the pretraining regime, which is the one documented
# deviation from identical treatment (§14.7).
NORM_MEAN, NORM_STD = ((IMAGENET_MEAN, IMAGENET_STD) if SPEC["pretrained"]
                       else (CFG["dataset_mean"], CFG["dataset_std"]))

print("model :", MODEL_NAME)
print("spec  :", SPEC)
print("norm  :", "ImageNet" if SPEC["pretrained"] else "dataset")

In [ ]:
def build_model(num_classes, pretrained=True):
    """
    EfficientNet-B0 with ImageNet weights, final classifier layer replaced.

    B0 is the baseline member of the compound-scaled family: depth, width and
    input resolution scaled jointly by a single coefficient (Tan & Le, 2019).
    """
    w = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    m = models.efficientnet_b0(weights=w)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    return m


def set_backbone_trainable(model, trainable):
    """
    Staged fine-tuning, identical in form to the ResNet50 schedule so that the
    two transfer-learning models differ only in backbone (§14.8).
    """
    for pn, p in model.named_parameters():
        p.requires_grad = trainable or pn.startswith("classifier")


def gradcam_target_layer(model):
    # Final conv stage before pooling.
    return [model.features[-1]]


_m = build_model(len(CFG["classes"]), pretrained=False)
print(f"{MODEL_NAME}: {sum(p.numel() for p in _m.parameters()):,} parameters")
del _m

## 9 · Training harness  ·  §14.8

Optimiser family (AdamW), schedule shape (cosine annealing), batch size, epoch
budget, early-stopping rule, class-weighted loss and seed are **identical across
all three notebooks**. Only the architecture and its pretraining regime vary.

Model selection uses validation **macro-F1**, never accuracy, so a model that
ignores the minority classes cannot win on the majority class alone. The test
partition is never touched here.

### Staged schedule for this model

**Staged fine-tuning, identical schedule to ResNet50.** Epochs 1–5 train the head alone at 1e-3 with the backbone frozen; from epoch 6 the backbone unfreezes at 1e-4. Using the same schedule for both pretrained models is what isolates the backbone as the only difference between them.

### Crash safety

`last.pt` is written **every epoch** with model, optimiser, scheduler and AMP
scaler state, and `history.csv` is flushed alongside it. An interruption — power
loss, a closed lid, a kernel restart — costs one epoch rather than the whole run.
Re-run with `resume=True` to continue.

In [ ]:
def run_epoch(model, loader, crit, opt, scaler, train):
    model.train() if train else model.eval()
    tot, n, P, G = 0.0, 0, [], []
    with (torch.enable_grad() if train else torch.no_grad()):
        for x, y in tqdm(loader, leave=False, desc="train" if train else "val"):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            if train:
                opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=scaler is not None):
                out = model(x); loss = crit(out, y)
            if train:
                if scaler is not None:
                    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                else:
                    loss.backward(); opt.step()
            tot += loss.item() * y.size(0); n += y.size(0)
            P.append(out.argmax(1).cpu()); G.append(y.cpu())
    p, g = torch.cat(P).numpy(), torch.cat(G).numpy()
    return tot / n, (p == g).mean(), f1_score(g, p, average="macro", zero_division=0)


def train_model(cfg, manifest_df, patch_root, tag="main", resume=True):
    set_seed(cfg["seed"])                          # identical init conditions
    classes = cfg["classes"]

    ds_tr = PatchDataset(manifest_df, patch_root, "train", classes,
                         build_transforms(cfg["image_size"], NORM_MEAN, NORM_STD, True))
    ds_va = PatchDataset(manifest_df, patch_root, "val", classes,
                         build_transforms(cfg["image_size"], NORM_MEAN, NORM_STD, False))
    kw = dict(batch_size=cfg["batch_size"], num_workers=NUM_WORKERS, pin_memory=True)
    dl_tr = DataLoader(ds_tr, shuffle=True, drop_last=True, **kw)
    dl_va = DataLoader(ds_va, shuffle=False, **kw)
    print(f"\n{'='*78}\n{MODEL_NAME}  |  train {len(ds_tr):,}  val {len(ds_va):,}\n{'='*78}")

    model = build_model(len(classes), SPEC["pretrained"]).to(DEVICE)

    # Class-weighted loss: same formula in all three notebooks (§14.5).
    counts = np.bincount(ds_tr.labels, minlength=len(classes))
    w = torch.tensor(counts.sum() / (len(classes) * np.maximum(counts, 1)),
                     dtype=torch.float32, device=DEVICE)
    print("class counts:", dict(zip(classes, counts.tolist())))
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=cfg["label_smoothing"])
    scaler = torch.amp.GradScaler("cuda") if cfg["amp"] and DEVICE.type == "cuda" else None

    outdir = cfg["runs_dir"] / tag / MODEL_NAME
    outdir.mkdir(parents=True, exist_ok=True)
    fe = SPEC["freeze_epochs"]
    best, best_ep, bad, hist, start_ep = -1.0, -1, 0, [], 1
    opt, sched = None, None

    # ---- resume from the crash-safe checkpoint --------------------------
    last_p = outdir / "last.pt"
    if resume and last_p.exists():
        ck = torch.load(last_p, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ck["model"])
        set_backbone_trainable(model, ck["epoch"] > fe)
        lr = SPEC["lr_head"] if ck["epoch"] <= fe else SPEC["lr_finetune"]
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                lr=lr, weight_decay=cfg["weight_decay"])
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=max(1, cfg["epochs"] - (fe + 1) + 1))
        opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        if scaler and ck.get("scaler"):
            scaler.load_state_dict(ck["scaler"])
        best, best_ep, bad = ck["best"], ck["best_ep"], ck["bad"]
        hist, start_ep = ck["hist"], ck["epoch"] + 1
        print(f"[resume] from epoch {ck['epoch']}, best so far {best:.4f}")

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    for ep in range(start_ep, cfg["epochs"] + 1):
        if opt is None or (fe and ep == fe + 1):           # stage boundary
            frozen = ep <= fe
            set_backbone_trainable(model, not frozen)
            lr = SPEC["lr_head"] if frozen else SPEC["lr_finetune"]
            opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                    lr=lr, weight_decay=cfg["weight_decay"])
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(
                opt, T_max=max(1, cfg["epochs"] - ep + 1))
            n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"[stage] epoch {ep}: backbone {'FROZEN' if frozen else 'UNFROZEN'}, "
                  f"lr={lr}, trainable={n_train:,}")

        tl, ta, tf = run_epoch(model, dl_tr, crit, opt, scaler, True)
        vl, va, vf = run_epoch(model, dl_va, crit, None, scaler, False)
        sched.step()
        peak = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0
        print(f"ep {ep:3d} | train loss {tl:.4f} f1 {tf:.4f} | "
              f"val loss {vl:.4f} acc {va:.4f} macroF1 {vf:.4f} | peak {peak:.2f} GB")
        hist.append(dict(epoch=ep, train_loss=tl, train_f1=tf, val_loss=vl,
                         val_acc=va, val_macro_f1=vf, lr=opt.param_groups[0]["lr"]))

        if vf > best:
            best, best_ep, bad = vf, ep, 0
            torch.save({"model": model.state_dict(), "epoch": ep,
                        "val_macro_f1": vf, "classes": classes}, outdir / "best.pt")
        else:
            bad += 1

        # crash-safe: flushed EVERY epoch, not only at the end
        pd.DataFrame(hist).to_csv(outdir / "history.csv", index=False)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
                    "sched": sched.state_dict(),
                    "scaler": scaler.state_dict() if scaler else None,
                    "epoch": ep, "best": best, "best_ep": best_ep,
                    "bad": bad, "hist": hist}, outdir / "last.pt")

        if bad >= cfg["early_stopping_patience"]:
            print(f"early stop at epoch {ep} (no improvement for {bad})")
            break

    el = time.time() - t0
    summary = dict(model=MODEL_NAME, best_epoch=best_ep, best_val_macro_f1=float(best),
                   training_time_sec=el, epochs_run=len(hist),
                   batch_size=cfg["batch_size"], amp=bool(scaler),
                   num_workers=NUM_WORKERS,
                   peak_gpu_gb=float(torch.cuda.max_memory_allocated() / 1e9)
                   if DEVICE.type == "cuda" else None)
    json.dump(summary, open(outdir / "train_summary.json", "w"), indent=2)
    print(f"\nBEST val macro-F1 {best:.4f} @ epoch {best_ep}  |  {el/60:.1f} min  -> {outdir}")
    return summary

print("training harness ready")

### 9.1 · Train

Close anything holding VRAM (animated wallpaper, browsers) before starting, and
set Windows to never sleep on AC power.

Already-trained runs are detected and skipped. Delete
`runs/main/<model>/train_summary.json` to force a retrain, or set
`FORCE_RETRAIN = True`.

In [ ]:
FORCE_RETRAIN = False

_sum_path = CFG["runs_dir"] / "main" / MODEL_NAME / "train_summary.json"
if _sum_path.exists() and not FORCE_RETRAIN:
    summary = json.load(open(_sum_path))
    print(f"already trained: best val macro-F1 {summary['best_val_macro_f1']:.4f} "
          f"@ epoch {summary['best_epoch']} ({summary['epochs_run']} epochs, "
          f"{summary['training_time_sec']/60:.1f} min)")
    print("set FORCE_RETRAIN = True to run again")
else:
    summary = train_model(CFG, manifest_df, PATCH_ROOT, resume=True)

torch.cuda.empty_cache()

### 9.2 · Learning curves

Three things to read off these plots: the kink where the backbone unfreezes;
whether validation loss **flattens** (benign) or **turns upward** (harmful
overfitting — the best checkpoint would then be mid-run and everything after it
wasted); and whether early stopping fired before the epoch budget ran out.

In [ ]:
h = pd.read_csv(CFG["runs_dir"] / "main" / MODEL_NAME / "history.csv")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, cols, title in [
        (axes[0], ["val_macro_f1", "train_f1"], "Macro-F1"),
        (axes[1], ["val_loss", "train_loss"], "Loss"),
        (axes[2], ["lr"], "Learning rate")]:
    for c in cols:
        ax.plot(h.epoch, h[c], marker="o", ms=3, label=c)
    if SPEC["freeze_epochs"]:
        ax.axvline(SPEC["freeze_epochs"] + 0.5, color="crimson", ls="--", lw=1,
                   label="backbone unfrozen")
    ax.set_xlabel("epoch"); ax.set_title(f"{MODEL_NAME} — {title}")
    ax.grid(alpha=.3); ax.legend(fontsize=8)
axes[2].set_yscale("log")
plt.tight_layout()
plt.savefig(CFG["reports_dir"] / f"curves_{MODEL_NAME}.png", dpi=150)
plt.show()

gap = h.train_f1.iloc[-1] - h.val_macro_f1.iloc[-1]
turn = h.val_loss.idxmin()
print(f"final train-val F1 gap : {gap:.4f}")
print(f"val loss minimum       : epoch {int(h.epoch[turn])} of {int(h.epoch.max())}")
print("val loss still falling at the end -> overfitting was benign"
      if turn >= len(h) - 3 else
      "val loss turned upward -> the gap cost real generalisation; report this")

## 10 · Evaluation and computational profiling  ·  §17.1–17.2

The **first and only** time this notebook touches the test partition.

Latency is measured after warm-up with explicit `torch.cuda.synchronize()`.
Without the sync you time kernel *launches* rather than kernel *execution*, and
the numbers come out absurdly fast. Parameter count, on-disk size, training time
and both batch-1 latency and batch-32 throughput are recorded, since §17.2 asks
for the accuracy-versus-cost frontier rather than accuracy alone.

In [ ]:
def profile_latency(model, size, batch=1, runs=200, warmup=50):
    model.eval()
    x = torch.randn(batch, 3, size, size, device=DEVICE)
    with torch.no_grad():
        for _ in range(warmup):
            model(x)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(runs):
            model(x)
        if DEVICE.type == "cuda": torch.cuda.synchronize()
        dt = time.perf_counter() - t0
    return dt / runs / batch * 1000, batch * runs / dt


def evaluate_model(cfg, manifest_df, patch_root, tag="main"):
    classes = cfg["classes"]
    ckdir = cfg["runs_dir"] / tag / MODEL_NAME
    ck = torch.load(ckdir / "best.pt", map_location=DEVICE, weights_only=False)
    model = build_model(len(classes), pretrained=False).to(DEVICE)
    model.load_state_dict(ck["model"]); model.eval()
    print(f"loaded best.pt from epoch {ck['epoch']} (val macro-F1 {ck['val_macro_f1']:.4f})")

    ds = PatchDataset(manifest_df, patch_root, "test", classes,
                      build_transforms(cfg["image_size"], NORM_MEAN, NORM_STD, False))
    dl = DataLoader(ds, batch_size=cfg["batch_size"], num_workers=NUM_WORKERS)

    P = []
    with torch.no_grad():
        for x, _ in tqdm(dl, desc=f"test:{MODEL_NAME}", leave=False):
            P.append(model(x.to(DEVICE)).argmax(1).cpu())
    p, g = torch.cat(P).numpy(), np.array(ds.labels)

    pr, rc, f1, sup = precision_recall_fscore_support(
        g, p, zero_division=0, labels=range(len(classes)))
    lat1, _ = profile_latency(model, cfg["image_size"], 1)
    _, thr32 = profile_latency(model, cfg["image_size"], 32, runs=50, warmup=20)
    ts = json.load(open(ckdir / "train_summary.json"))

    res = dict(model=MODEL_NAME,
               accuracy=accuracy_score(g, p),
               macro_f1=f1_score(g, p, average="macro", zero_division=0),
               weighted_f1=f1_score(g, p, average="weighted", zero_division=0),
               macro_precision=float(pr.mean()), macro_recall=float(rc.mean()),
               trainable_params=sum(q.numel() for q in model.parameters()),
               model_size_mb=round((ckdir / "best.pt").stat().st_size / 1e6, 2),
               latency_ms_bs1=round(lat1, 3), throughput_bs32=round(thr32, 1),
               training_time_min=round(ts["training_time_sec"] / 60, 1),
               best_epoch=ts["best_epoch"], best_val_macro_f1=ts["best_val_macro_f1"],
               peak_gpu_gb=ts.get("peak_gpu_gb"), n_test=len(g))
    for i, c in enumerate(classes):
        res[f"f1_{c}"] = round(float(f1[i]), 4)
        res[f"precision_{c}"] = round(float(pr[i]), 4)
        res[f"recall_{c}"] = round(float(rc[i]), 4)
        res[f"support_{c}"] = int(sup[i])

    print(f"\n{'='*78}\n{MODEL_NAME} — held-out test set\n{'='*78}")
    print(classification_report(g, p, target_names=classes, zero_division=0, digits=4))
    return res, confusion_matrix(g, p, labels=range(len(classes))), (g, p, ds)


result, cm, raw = evaluate_model(CFG, manifest_df, PATCH_ROOT)

# Per-country breakdown. Cells with small support are noise, so the count is
# printed alongside every score rather than the score alone.
g, p, ds = raw
per_country = pd.DataFrame({"country": ds.countries, "true": g, "pred": p})
rows = []
for c, sub in per_country.groupby("country"):
    rows.append(dict(country=c, n=len(sub),
                     accuracy=round((sub.true == sub.pred).mean(), 4),
                     macro_f1=round(f1_score(sub.true, sub.pred,
                                             average="macro", zero_division=0), 4)))
pc = pd.DataFrame(rows).sort_values("n", ascending=False)
print("\nPer-country test performance (n = test patches; small n is unreliable):")
print(pc.to_string(index=False))
pc.to_csv(CFG["reports_dir"] / f"per_country_{MODEL_NAME}.csv", index=False)

# Save this model's result so the three notebooks can be merged in Section 12.
pd.DataFrame([result]).to_csv(CFG["reports_dir"] / f"result_{MODEL_NAME}.csv", index=False)
pd.DataFrame(cm, index=CFG["classes"], columns=CFG["classes"]).to_csv(
    CFG["reports_dir"] / f"confusion_{MODEL_NAME}.csv")
print(f"\nsaved -> reports/result_{MODEL_NAME}.csv")

### 10.1 · Confusion matrix

In [ ]:
# Row-normalised so minority classes stay legible beside the majority class.
fig, ax = plt.subplots(figsize=(5.6, 5))
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(CFG["classes"])), CFG["classes"])
ax.set_yticks(range(len(CFG["classes"])), CFG["classes"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"{MODEL_NAME} (row-normalised)")
for i in range(len(CFG["classes"])):
    for j in range(len(CFG["classes"])):
        ax.text(j, i, f"{cmn[i,j]:.2f}\n({cm[i,j]})", ha="center", va="center",
                fontsize=8, color="white" if cmn[i, j] > .5 else "black")
fig.colorbar(im); plt.tight_layout()
plt.savefig(CFG["reports_dir"] / f"confusion_{MODEL_NAME}.png", dpi=150)
plt.show()

off = [(CFG["classes"][i], CFG["classes"][j], cm[i, j], cmn[i, j])
       for i in range(len(CFG["classes"])) for j in range(len(CFG["classes"])) if i != j]
off.sort(key=lambda t: -t[3])
print("Largest confusions (true -> predicted):")
for t, pdc, n, r in off[:3]:
    print(f"  {t} -> {pdc}: {r*100:.1f}% ({n} patches)")
print("\nD20 -> D00 is the expected dominant error: alligator cracking develops")
print("from interconnected longitudinal cracks, so early-stage D20 genuinely")
print("contains D00 morphology. Part of this error may be label ambiguity")
print("rather than model failure — inspect a few of these patches by eye.")

## 11 · Grad-CAM diagnostic analysis  ·  §18

Samples are **stratified across every class and across both correct and incorrect
predictions**, so the figure cannot become an unrepresentative selection of
flattering examples. The purpose is diagnostic, not decorative.

Target layer for this model: `model.features[-1]`.

Read the output against §18's questions. Look specifically for activation on lane
markings, vegetation, image borders or country-specific artefacts rather than on
the pavement — that would be evidence of a shortcut feature, and it is the most
interesting thing this analysis can reveal. For misclassified patches, ask whether
the model looked in the wrong place or looked in the right place and still could
not discriminate.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def gradcam_grid(cfg, n_per_cell=4, tag="main"):
    classes = cfg["classes"]
    ck = torch.load(cfg["runs_dir"] / tag / MODEL_NAME / "best.pt",
                    map_location=DEVICE, weights_only=False)
    model = build_model(len(classes), pretrained=False).to(DEVICE)
    model.load_state_dict(ck["model"]); model.eval()

    g, p, ds = raw
    correct = p == g
    rng = np.random.default_rng(cfg["seed"])
    cam = GradCAM(model=model, target_layers=gradcam_target_layer(model))

    rows = []
    for ci, cn in enumerate(classes):
        for ok in (True, False):
            idx = np.where((g == ci) & (correct == ok))[0]
            if len(idx):
                rows.append((cn, ok, rng.choice(idx, min(n_per_cell, len(idx)),
                                                replace=False)))

    fig, axes = plt.subplots(len(rows), n_per_cell,
                             figsize=(2.4 * n_per_cell, 2.6 * len(rows)), squeeze=False)
    for r, (cn, ok, idxs) in enumerate(rows):
        for c in range(n_per_cell):
            ax = axes[r][c]; ax.set_xticks([]); ax.set_yticks([])
            if c >= len(idxs):
                ax.axis("off"); continue
            i = int(idxs[c])
            x, _ = ds[i]
            heat = cam(input_tensor=x.unsqueeze(0).to(DEVICE),
                       targets=[ClassifierOutputTarget(int(p[i]))])[0]
            base = np.asarray(Image.open(Path(PATCH_ROOT) / ds.paths[i]).convert("RGB")
                              .resize((cfg["image_size"],) * 2)) / 255.
            ax.imshow(np.clip(.55 * base + .45 * plt.get_cmap("jet")(heat)[..., :3], 0, 1))
            ax.set_title(f"{'correct' if ok else '-> ' + classes[p[i]]} · "
                         f"{ds.countries[i][:12]}", fontsize=6.5)
        axes[r][0].set_ylabel(f"{cn}\n{'hit' if ok else 'miss'}", fontsize=8)
    fig.suptitle(f"Grad-CAM · stratified test sample · {MODEL_NAME}", fontsize=12)
    plt.tight_layout()
    out = cfg["reports_dir"] / "gradcam"; out.mkdir(exist_ok=True)
    plt.savefig(out / f"gradcam_{MODEL_NAME}.png", dpi=150)
    plt.show()

gradcam_grid(CFG)

## 12 · Merge the three results

Each notebook writes `reports/result_<model>.csv`. Once all three exist — whoever
finishes last, or after collecting the files from the other two students — this
cell builds the comparison table and the accuracy-versus-cost plot that
Experiments A–C require.

It also re-checks the fingerprints. **If they do not all match, the comparison is
not valid** and the mismatch must be resolved before any conclusion is drawn.

Currently waiting on: `custom_cnn`, `resnet50` (plus this notebook's own
result, written in Section 10).

In [ ]:
ALL_MODELS = ["custom_cnn", "resnet50", "efficientnet_b0"]

# ---- fingerprint cross-check ------------------------------------------------
fps = {}
for m in ALL_MODELS:
    fp = CFG["reports_dir"] / f"fingerprint_{m}.json"
    if fp.exists():
        fps[m] = json.load(open(fp))

print("=" * 78)
print("FINGERPRINT CROSS-CHECK")
print("=" * 78)
for m in ALL_MODELS:
    if m in fps:
        print(f"  {m:18s} data={fps[m]['data_fingerprint']} "
              f"protocol={fps[m]['protocol_fingerprint']} "
              f"n={fps[m]['n_patches']:,}")
    else:
        print(f"  {m:18s} (not available yet)")

if len(fps) >= 2:
    uniq = {v["data_fingerprint"] for v in fps.values()}
    print(f"\n  -> {'MATCH — preprocessing identical' if len(uniq) == 1 else
                   '*** MISMATCH — results are NOT comparable ***'}")

# ---- merge results ----------------------------------------------------------
found = {}
for m in ALL_MODELS:
    f = CFG["reports_dir"] / f"result_{m}.csv"
    if f.exists():
        found[m] = pd.read_csv(f).iloc[0]

print(f"\nresults available: {len(found)}/3 -> {list(found)}")

if len(found) < 3:
    print("Run the other notebooks (or collect their reports/result_*.csv) "
          "and re-run this cell.")
else:
    comparison = pd.DataFrame([found[m] for m in ALL_MODELS]).reset_index(drop=True)
    comparison.to_csv(CFG["reports_dir"] / "comparison.csv", index=False)

    print("\n" + "=" * 78 + "\nHEADLINE COMPARISON\n" + "=" * 78)
    print(comparison[["model", "accuracy", "macro_f1", "weighted_f1",
                      "trainable_params", "model_size_mb", "latency_ms_bs1",
                      "training_time_min"]].to_string(index=False))

    print("\nPer-class F1 — the table that answers the research question:")
    print(comparison[["model"] + [f"f1_{c}" for c in CFG["classes"]]].to_string(index=False))

    # accuracy-versus-cost frontier (§17.2)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    cols = ["#c0392b", "#2980b9", "#27ae60"]
    for ax, xc, xl, logx in [
            (axes[0], "trainable_params", "Trainable parameters", True),
            (axes[1], "latency_ms_bs1", "Inference latency, ms (bs=1)", False)]:
        ax.scatter(comparison[xc], comparison.macro_f1, s=120, c=cols)
        for _, r in comparison.iterrows():
            ax.annotate(r["model"], (r[xc], r["macro_f1"]),
                        textcoords="offset points", xytext=(7, 7), fontsize=9)
        if logx:
            ax.set_xscale("log")
        ax.set_xlabel(xl); ax.set_ylabel("Test macro-F1"); ax.grid(alpha=.3)
    plt.suptitle("Accuracy versus computational cost (§17.2)")
    plt.tight_layout()
    plt.savefig(CFG["reports_dir"] / "accuracy_vs_cost.png", dpi=150)
    plt.show()

    # combined confusion matrices
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
    for ax, m in zip(axes, ALL_MODELS):
        cf = CFG["reports_dir"] / f"confusion_{m}.csv"
        if not cf.exists():
            ax.axis("off"); continue
        c_ = pd.read_csv(cf, index_col=0).values
        cn_ = c_ / np.maximum(c_.sum(1, keepdims=True), 1)
        ax.imshow(cn_, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(CFG["classes"])), CFG["classes"])
        ax.set_yticks(range(len(CFG["classes"])), CFG["classes"])
        ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(m)
        for i in range(len(CFG["classes"])):
            for j in range(len(CFG["classes"])):
                ax.text(j, i, f"{cn_[i,j]:.2f}", ha="center", va="center",
                        fontsize=7, color="white" if cn_[i, j] > .5 else "black")
    plt.tight_layout()
    plt.savefig(CFG["reports_dir"] / "confusion_matrices.png", dpi=150)
    plt.show()

    eff = comparison.sort_values("trainable_params")
    print(f"\nSmallest model: {eff.iloc[0]['model']} "
          f"({eff.iloc[0]['trainable_params']:,.0f} params, "
          f"macro-F1 {eff.iloc[0]['macro_f1']:.4f})")
    print(f"Largest model : {eff.iloc[-1]['model']} "
          f"({eff.iloc[-1]['trainable_params']:,.0f} params, "
          f"macro-F1 {eff.iloc[-1]['macro_f1']:.4f})")

## 13 · Reproducibility record

Paste this into the Phase 2 report. §17.2 requires that all efficiency figures be
measured on a single documented hardware and software stack, and that
cross-machine comparisons are not made.

**If the three models were trained on different machines, their `training_time`
and `latency` figures are not comparable** and only parameter count and model size
should be used for the efficiency argument. Record each machine separately and say
so explicitly.

In [ ]:
print("model        :", MODEL_NAME)
print("date         :", time.strftime("%Y-%m-%d %H:%M"))
print("python       :", platform.python_version())
print("platform     :", platform.platform())
print("torch        :", torch.__version__, "| CUDA", torch.version.cuda)
print("gpu          :", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU")
print("-" * 60)
print("seed         :", CFG["seed"])
print("classes      :", CFG["classes"])
print("image_size   :", CFG["image_size"], "| batch", CFG["batch_size"],
      "| AMP", CFG["amp"], "| workers", NUM_WORKERS)
print("epochs       :", CFG["epochs"], "| patience", CFG["early_stopping_patience"])
print("group_size   :", CFG["group_size"], "| ratios", CFG["ratios"])
print("min_side     :", CFG["min_side"], "| max_aspect", CFG["max_aspect"],
      "| margin", CFG["margin"])
print("normalisation:", "ImageNet" if SPEC["pretrained"] else "dataset",
      NORM_MEAN, NORM_STD)
print("spec         :", SPEC)
print("-" * 60)
print("data fingerprint:", data_fp, "| protocol:", proto_fp)